# Phase 1 - Tóm tắt chất lượng dữ liệu và thống kê cơ bản
Notebook này tổng hợp số dòng, dải ngày, và các chỉ số chính sau khi ETL hoàn tất.

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED = ROOT / 'data' / 'processed'
INTERIM = ROOT / 'data' / 'interim'

tables = {
    'fact_transactions': pd.read_parquet(PROCESSED / 'fact_transactions.parquet'),
    'agg_funnel_monthly': pd.read_parquet(PROCESSED / 'agg_funnel_monthly.parquet'),
    'fact_inventory': pd.read_parquet(PROCESSED / 'fact_inventory.parquet'),
    'dim_users': pd.read_parquet(PROCESSED / 'dim_users.parquet'),
    'dim_products': pd.read_parquet(PROCESSED / 'dim_products.parquet'),
    'dim_dcs': pd.read_parquet(PROCESSED / 'dim_dcs.parquet'),
    'dim_date': pd.read_parquet(PROCESSED / 'dim_date.parquet'),
}

summary = pd.DataFrame([
    {'table': name, 'rows': len(df), 'columns': df.shape[1], 'memory_mb': round(df.memory_usage(deep=True).sum() / (1024**2), 3)}
    for name, df in tables.items()
])
summary

In [ ]:
date_stats = pd.DataFrame([
    {'table': 'fact_transactions', 'min_date': pd.to_datetime(tables['fact_transactions']['created_at']).min(), 'max_date': pd.to_datetime(tables['fact_transactions']['created_at']).max()},
    {'table': 'agg_funnel_monthly', 'min_date': pd.to_datetime(tables['agg_funnel_monthly']['year_month']).min(), 'max_date': pd.to_datetime(tables['agg_funnel_monthly']['year_month']).max()},
    {'table': 'fact_inventory', 'min_date': pd.to_datetime(tables['fact_inventory']['created_at']).min() if 'created_at' in tables['fact_inventory'].columns else pd.NaT, 'max_date': pd.to_datetime(tables['fact_inventory']['created_at']).max() if 'created_at' in tables['fact_inventory'].columns else pd.NaT},
    {'table': 'dim_date', 'min_date': pd.to_datetime(tables['dim_date']['date']).min(), 'max_date': pd.to_datetime(tables['dim_date']['date']).max()},
])
date_stats

In [ ]:
quality_summary_path = INTERIM / 'phase1_quality_summary.json'
if quality_summary_path.exists():
    quality_summary = pd.read_json(quality_summary_path, typ='series')
    print('Quality summary loaded from data/interim/phase1_quality_summary.json')
    print(quality_summary)
else:
    print('No quality summary file found.')

In [ ]:
revenue = float(tables['fact_transactions']['sale_price'].sum())
gross_profit = float(tables['fact_transactions']['gross_profit'].sum())
avg_order_value = float(tables['fact_transactions'].groupby('order_id')['sale_price'].sum().mean())
sell_through_rate = float(tables['fact_inventory']['is_sold'].mean() * 100)

metrics = pd.DataFrame([
    {'metric': 'total_revenue', 'value': revenue},
    {'metric': 'gross_profit', 'value': gross_profit},
    {'metric': 'avg_order_value', 'value': avg_order_value},
    {'metric': 'sell_through_rate_pct', 'value': sell_through_rate},
])
metrics

## Kết luận nhanh
- Dữ liệu đã được load, làm sạch, và xuất ra Parquet thành công.
- Báo cáo chất lượng và bot candidates đã được lưu trong `data/interim/`.
- Notebook này dùng làm điểm kiểm tra nhanh cho Phase 1 trước khi sang phân tích sâu hơn.